# Modules 3 & 4 --- DataLoaders, EDA, KNN and K-Means

**Course:** Python and Machine Learning with PyTorch
**Companion slides:** `Presentation_3_Dataloaders_EDA.pdf` and `Presentation_4_KNN_KMeans.pdf`

---

### What you will do here

1. **Setup** --- libraries, imports, device.
2. **Theory in practice**
   - **Part A** --- images: `torchvision` datasets, transforms, a custom `Dataset`, and `DataLoader`.
   - **Part B** --- CSV: pandas to tensors, a `CSVDataset` class, correct normalisation.
   - **Part C** --- EDA: batch statistics, image grids, correlation matrices.
   - **Part D** --- KNN implemented from scratch in PyTorch, with decision boundaries.
   - **Part E** --- K-Means implemented from scratch, with the elbow method.
3. **Challenge** --- k-means++ initialisation, and a weighted KNN.

scikit-learn appears **only** to generate datasets and to check metrics. Every
algorithm is written with PyTorch tensors.

## 1. Setup

In [ ]:
!pip install torch torchvision matplotlib pandas scikit-learn --quiet

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.utils import make_grid

# sklearn: datasets and metrics ONLY
from sklearn.datasets import make_classification, make_blobs, load_iris
from sklearn.metrics import accuracy_score, adjusted_rand_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("device     :", device)

## 2. Theory in Practice

### Part A --- Images: `Dataset`, `transforms`, `DataLoader`

`transforms.ToTensor()` converts a PIL image to a `(C, H, W)` float tensor scaled
to `[0, 1]`. `Normalize` then shifts and scales it, so **`ToTensor` must come
first**.

In [ ]:
# ---- user-adjustable parameters ------------------------------------------
BATCH_SIZE  = 64
NUM_WORKERS = 2       # keep this at 2 in Colab; larger values often crash
# --------------------------------------------------------------------------

# MNIST statistics, precomputed on the training split by convention
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

train_tf = transforms.Compose([
    transforms.RandomRotation(8),                  # augmentation, train only
    transforms.ToTensor(),                         # PIL -> tensor in [0, 1]
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
])

test_tf = transforms.Compose([                     # no augmentation at test time
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
])

train_set = datasets.MNIST(root="./data", train=True,  download=True, transform=train_tf)
test_set  = datasets.MNIST(root="./data", train=False, download=True, transform=test_tf)

print("train samples:", len(train_set))
print("test  samples:", len(test_set))
print("classes      :", train_set.classes)

In [ ]:
# A Dataset supports len() and indexing. That is the whole contract.
img, label = train_set[0]

print("type(train_set[0]) :", type(train_set[0]))
print("image shape        :", tuple(img.shape), " -> (C, H, W)")
print("image dtype        :", img.dtype)
print("label              :", label, type(label))
print("value range        :", f"[{img.min():.3f}, {img.max():.3f}]  (normalised, so not [0,1])")

In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False,
                          num_workers=NUM_WORKERS)

print("batches per epoch (train):", len(train_loader), "=", len(train_set), "/", BATCH_SIZE)
print("batches per epoch (test) :", len(test_loader))

images, labels = next(iter(train_loader))
print("\none batch:")
print("  images:", tuple(images.shape), " -> (N, C, H, W)")
print("  labels:", tuple(labels.shape), labels[:10].tolist())

In [ ]:
# Why shuffle=True matters: without it, batches can be sorted by class.
sorted_loader = DataLoader(train_set, batch_size=16, shuffle=False)
shuffled_loader = DataLoader(train_set, batch_size=16, shuffle=True)

_, lab_sorted = next(iter(sorted_loader))
_, lab_shuffled = next(iter(shuffled_loader))

print("shuffle=False labels:", lab_sorted.tolist())
print("shuffle=True  labels:", lab_shuffled.tolist())

#### A custom `Dataset` over the same data

`ImageFolder` handles the folder-per-class layout. When your data does not fit
that shape, you write the class yourself. Only `__len__` and `__getitem__` are
required. Note that `__init__` stores **references**, and the actual work happens
lazily in `__getitem__`.

In [ ]:
class MNISTWrapper(Dataset):
    """A custom Dataset that wraps raw MNIST tensors.

    Demonstrates the contract without needing a folder of JPEGs.
    """

    def __init__(self, data, targets, transform=None, keep=None):
        # data: uint8 tensor (N, 28, 28); targets: int64 tensor (N,)
        if keep is not None:
            data, targets = data[:keep], targets[:keep]
        self.data = data
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        img = self.data[idx].float() / 255.0      # (28, 28) in [0, 1]
        img = img.unsqueeze(0)                    # add the channel dim -> (1, 28, 28)
        if self.transform is not None:
            img = self.transform(img)
        return img, int(self.targets[idx])


normalise = transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))

custom_set = MNISTWrapper(train_set.data, train_set.targets,
                          transform=normalise, keep=10000)

print("custom dataset length:", len(custom_set))
cimg, clabel = custom_set[0]
print("custom sample shape  :", tuple(cimg.shape), "| label:", clabel)

# Confirm it produces the same thing as torchvision's own pipeline
tv_img, tv_label = datasets.MNIST(root="./data", train=True, transform=test_tf)[0]
print("\nsame label as torchvision:", clabel == tv_label)
print("max absolute difference  :", (cimg - tv_img).abs().max().item())

In [ ]:
# A custom Dataset drops straight into a DataLoader, no extra work.
custom_loader = DataLoader(custom_set, batch_size=32, shuffle=True, num_workers=0)

cx, cy = next(iter(custom_loader))
print("custom loader batch:", tuple(cx.shape), tuple(cy.shape))
print("labels:", cy[:10].tolist())

In [ ]:
# Effect of num_workers on wall time. Results vary between runs and runtimes.
def time_one_epoch(loader, max_batches=60):
    start = time.time()
    for i, _ in enumerate(loader):
        if i >= max_batches:
            break
    return time.time() - start

for nw in [0, 2]:
    ld = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=nw)
    print(f"num_workers={nw}: {time_one_epoch(ld):.2f} s for 60 batches")

### Part B --- CSV data: pandas to a `CSVDataset`

We load Iris, write it to disk as a real CSV, then read it back the way you would
with your own data.

The important subtlety: normalisation statistics must be computed on the
**training split only**. Using the full dataset leaks test information into
training.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.columns = [c.replace(" (cm)", "").replace(" ", "_") for c in df.columns]

CSV_PATH = "iris.csv"
df.to_csv(CSV_PATH, index=False)
print("wrote", CSV_PATH)

df.head()

In [ ]:
FEATURE_COLS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET_COL   = "target"

print("shape        :", df.shape)
print("missing values:\n", df.isna().sum().to_string())
print("\nclass balance:\n", df[TARGET_COL].value_counts().sort_index().to_string())
print("\nsummary statistics:")
df[FEATURE_COLS].describe().round(3)

In [ ]:
class CSVDataset(Dataset):
    """Tabular Dataset with optional standardisation.

    Pass mean/std from the TRAINING split when building the validation set,
    so that no test information leaks into training.
    """

    def __init__(self, csv_path, feature_cols, target_col,
                 indices=None, mean=None, std=None, normalize=True):
        frame = pd.read_csv(csv_path)
        if indices is not None:
            frame = frame.iloc[indices]

        X = torch.tensor(frame[feature_cols].values, dtype=torch.float32)
        y = torch.tensor(frame[target_col].values, dtype=torch.long)

        if normalize:
            self.mean = X.mean(dim=0) if mean is None else mean
            self.std  = X.std(dim=0)  if std  is None else std
            X = (X - self.mean) / (self.std + 1e-8)     # epsilon avoids /0
        else:
            self.mean, self.std = None, None

        self.X, self.y = X, y
        self.feature_cols = feature_cols

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Split indices FIRST, then build the two datasets
n = len(df)
perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED)).tolist()
split = int(0.8 * n)
train_idx, val_idx = perm[:split], perm[split:]

iris_train = CSVDataset(CSV_PATH, FEATURE_COLS, TARGET_COL, indices=train_idx)
iris_val   = CSVDataset(CSV_PATH, FEATURE_COLS, TARGET_COL, indices=val_idx,
                        mean=iris_train.mean, std=iris_train.std)   # reuse train stats

print("train:", len(iris_train), "| val:", len(iris_val))
print("\ntraining means used for scaling:", iris_train.mean.round(decimals=3).tolist())
print("training stds  used for scaling:", iris_train.std.round(decimals=3).tolist())
print("\ntrain set column means after scaling (should be ~0):",
      iris_train.X.mean(dim=0).round(decimals=4).tolist())
print("val set column means after scaling (NOT exactly 0, and that is correct):",
      iris_val.X.mean(dim=0).round(decimals=4).tolist())

In [ ]:
iris_train_loader = DataLoader(iris_train, batch_size=16, shuffle=True)
iris_val_loader   = DataLoader(iris_val,   batch_size=32, shuffle=False)

bx, by = next(iter(iris_train_loader))
print("batch features:", tuple(bx.shape))
print("batch labels  :", by.tolist())

# random_split is the shortcut when you do not need separate statistics
whole = CSVDataset(CSV_PATH, FEATURE_COLS, TARGET_COL)
a, b = random_split(whole, [120, 30],
                    generator=torch.Generator().manual_seed(SEED))
print("\nrandom_split gives:", len(a), "and", len(b))

### Part C --- Exploratory Data Analysis

Never train on data you have not looked at. Work through the checklist from the
slides.

In [ ]:
images, labels = next(iter(train_loader))

print("--- EDA checklist on an image batch ---")
print("shape        :", tuple(images.shape))
print("dtype        :", images.dtype)
print("device       :", images.device)
print("min / max    :", f"{images.min():.3f} / {images.max():.3f}")
print("mean / std   :", f"{images.mean():.4f} / {images.std():.4f}")
print("per-channel mean:", images.mean(dim=[0, 2, 3]).tolist())
print("per-channel std :", images.std(dim=[0, 2, 3]).tolist())
print("any NaN      :", torch.isnan(images).any().item())
print("any Inf      :", torch.isinf(images).any().item())
print("label counts :", torch.bincount(labels, minlength=10).tolist())

In [ ]:
# Visualise a grid. make_grid returns (C, H, W); matplotlib wants (H, W, C).
grid = make_grid(images[:32], nrow=8, normalize=True, padding=2)

plt.figure(figsize=(10, 5))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray")
plt.axis("off")
plt.title("labels: " + " ".join(str(l) for l in labels[:32].tolist()), fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Class balance across the whole training set, and the pixel intensity histogram.
all_labels = train_set.targets
counts = torch.bincount(all_labels)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

axes[0].bar(range(10), counts.numpy(), color="steelblue")
axes[0].set_title("MNIST class balance"); axes[0].set_xlabel("digit")
axes[0].set_xticks(range(10)); axes[0].grid(alpha=0.3, axis="y")

raw_pixels = (train_set.data[:2000].float() / 255.0).reshape(-1)
axes[1].hist(raw_pixels.numpy(), bins=50, color="darkorange")
axes[1].set_title("raw pixel intensities (before Normalize)")
axes[1].set_yscale("log"); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print("most common digit :", counts.argmax().item(), "with", counts.max().item(), "samples")
print("least common digit:", counts.argmin().item(), "with", counts.min().item(), "samples")
print("imbalance ratio   :", round((counts.max() / counts.min()).item(), 3))

In [ ]:
# EDA on the tabular data, using tensor operations rather than pandas.
X_iris = torch.tensor(df[FEATURE_COLS].values, dtype=torch.float32)
y_iris = torch.tensor(df[TARGET_COL].values, dtype=torch.long)

print("column means:", X_iris.mean(dim=0).round(decimals=3).tolist())
print("column stds :", X_iris.std(dim=0).round(decimals=3).tolist())
print("column mins :", X_iris.min(dim=0).values.tolist())
print("column maxs :", X_iris.max(dim=0).values.tolist())

# Correlation matrix, built from scratch with broadcasting
Xc = X_iris - X_iris.mean(dim=0)
cov = (Xc.T @ Xc) / (X_iris.shape[0] - 1)
std = X_iris.std(dim=0)
corr = cov / (std[:, None] * std[None, :])       # outer product, shape (4, 4)

print("\ncorrelation matrix:")
print(pd.DataFrame(corr.numpy(), index=FEATURE_COLS, columns=FEATURE_COLS).round(3))
print("\nmatches pandas:", np.allclose(corr.numpy(), df[FEATURE_COLS].corr().values, atol=1e-5))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

# 1. Histogram of one feature, split by class
for c in range(3):
    axes[0].hist(X_iris[y_iris == c, 2].numpy(), bins=15, alpha=0.6,
                 label=iris.target_names[c])
axes[0].set_title("petal_length by class"); axes[0].legend(); axes[0].grid(alpha=0.3)

# 2. Scatter of two features
sc = axes[1].scatter(X_iris[:, 2], X_iris[:, 3], c=y_iris, cmap="viridis", s=25)
axes[1].set_xlabel("petal_length"); axes[1].set_ylabel("petal_width")
axes[1].set_title("the two most separable features"); axes[1].grid(alpha=0.3)

# 3. Correlation heatmap
im = axes[2].imshow(corr.numpy(), cmap="coolwarm", vmin=-1, vmax=1)
axes[2].set_xticks(range(4)); axes[2].set_yticks(range(4))
axes[2].set_xticklabels(FEATURE_COLS, rotation=45, ha="right", fontsize=7)
axes[2].set_yticklabels(FEATURE_COLS, fontsize=7)
axes[2].set_title("feature correlation")
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.tight_layout(); plt.show()

### Part D --- KNN from scratch in PyTorch

Recall the algorithm:

1. **Fit** is just storing the training set.
2. **Predict**: compute all distances, take the `k` smallest, vote.

`torch.cdist(A, B)` returns a matrix `D` where `D[i, j]` is the distance between
`A[i]` and `B[j]`. That single call replaces two nested loops.

In [ ]:
# ---- user-adjustable parameters ------------------------------------------
K = 5              # number of neighbours
P_NORM = 2         # 2 = Euclidean, 1 = Manhattan
N_POINTS = 400
# --------------------------------------------------------------------------

X_np, y_np = make_classification(
    n_samples=N_POINTS, n_features=2, n_informative=2, n_redundant=0,
    n_classes=3, n_clusters_per_class=1, class_sep=1.3,
    flip_y=0.03, random_state=SEED,
)

X_knn = torch.tensor(X_np, dtype=torch.float32)
y_knn = torch.tensor(y_np, dtype=torch.long)

# Standardise: KNN is a distance method, so feature scale changes the answer.
X_knn = (X_knn - X_knn.mean(dim=0)) / X_knn.std(dim=0)

n_tr = int(0.75 * len(X_knn))
perm = torch.randperm(len(X_knn), generator=torch.Generator().manual_seed(SEED))
tr_i, te_i = perm[:n_tr], perm[n_tr:]

Xtr, ytr = X_knn[tr_i].to(device), y_knn[tr_i].to(device)
Xte, yte = X_knn[te_i].to(device), y_knn[te_i].to(device)

print("train:", tuple(Xtr.shape), "| test:", tuple(Xte.shape))
print("class counts:", torch.bincount(y_knn).tolist())

In [ ]:
# First, understand torch.cdist on a tiny example.
A = torch.tensor([[0., 0.], [1., 1.]])
B = torch.tensor([[0., 0.], [3., 4.], [1., 0.]])

D = torch.cdist(A, B, p=2)
print("A shape:", tuple(A.shape), " B shape:", tuple(B.shape))
print("D shape:", tuple(D.shape), " -> D[i,j] = distance(A[i], B[j])")
print("D =\n", D)
print("\ncheck: distance((0,0),(3,4)) should be 5.0 ->", D[0, 1].item())

# The same thing by hand, with broadcasting
diff = A.unsqueeze(1) - B.unsqueeze(0)          # (2,1,2) - (1,3,2) -> (2,3,2)
D_manual = diff.pow(2).sum(dim=2).sqrt()
print("\nmanual version matches:", torch.allclose(D, D_manual, atol=1e-6))
print("Manhattan (p=1):\n", torch.cdist(A, B, p=1))

In [ ]:
class TorchKNN:
    """K-Nearest Neighbours, implemented with PyTorch tensors only."""

    def __init__(self, k=5, p=2):
        self.k = k
        self.p = p

    def fit(self, X, y):
        """Training is nothing more than storing the data."""
        self.X = X
        self.y = y
        self.n_classes = int(y.max().item()) + 1
        return self

    def _vote(self, labels):
        """labels: (m, k) -> (m,) majority label per row."""
        m = labels.shape[0]
        votes = torch.zeros(m, self.n_classes, device=labels.device)
        votes.scatter_add_(1, labels, torch.ones_like(labels, dtype=votes.dtype))
        return votes.argmax(dim=1)

    def predict(self, Q, chunk=4096):
        """Predict labels for Q, processing in chunks to bound memory."""
        out = []
        for start in range(0, Q.shape[0], chunk):
            q = Q[start:start + chunk]
            D = torch.cdist(q, self.X, p=self.p)                  # (chunk, n)
            idx = D.topk(self.k, dim=1, largest=False).indices    # (chunk, k)
            out.append(self._vote(self.y[idx]))
        return torch.cat(out)

    def score(self, Q, y_true):
        return (self.predict(Q) == y_true).float().mean().item()


knn = TorchKNN(k=K, p=P_NORM).fit(Xtr, ytr)

print(f"k = {K}, p = {P_NORM}")
print("train accuracy:", round(knn.score(Xtr, ytr), 4))
print("test  accuracy:", round(knn.score(Xte, yte), 4))

# Cross-check the metric with sklearn (metric only, not the model)
preds = knn.predict(Xte)
print("sklearn agrees:", round(accuracy_score(yte.cpu().numpy(), preds.cpu().numpy()), 4))

In [ ]:
# k = 1 must give perfect TRAINING accuracy: each point is its own neighbour.
knn1 = TorchKNN(k=1).fit(Xtr, ytr)
print("k=1 train accuracy:", round(knn1.score(Xtr, ytr), 4), " <- always 1.0, that is overfitting")
print("k=1 test  accuracy:", round(knn1.score(Xte, yte), 4))

In [ ]:
# Sweep k and compare the two distance metrics.
k_values = [1, 3, 5, 7, 9, 11, 15, 21, 31, 45]
results = {1: [], 2: []}

for p in (1, 2):
    for k in k_values:
        results[p].append(TorchKNN(k=k, p=p).fit(Xtr, ytr).score(Xte, yte))

plt.figure(figsize=(6.5, 3.6))
plt.plot(k_values, results[2], "o-", label="Euclidean (p=2)")
plt.plot(k_values, results[1], "s--", label="Manhattan (p=1)")
plt.xlabel("k"); plt.ylabel("test accuracy")
plt.title("choosing k"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

best_i = int(np.argmax(results[2]))
print(f"best Euclidean k = {k_values[best_i]} with accuracy {results[2][best_i]:.4f}")

#### Decision boundaries

Classify every point of a dense grid, then paint the regions. This is the picture
from the slides, now produced from your own implementation.

In [ ]:
def plot_knn_boundary(ax, model, X, y, title, resolution=250):
    x_min, x_max = X[:, 0].min().item() - 0.6, X[:, 0].max().item() + 0.6
    y_min, y_max = X[:, 1].min().item() - 0.6, X[:, 1].max().item() + 0.6

    xs = torch.linspace(x_min, x_max, resolution)
    ys = torch.linspace(y_min, y_max, resolution)
    xx, yy = torch.meshgrid(xs, ys, indexing="xy")

    grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1).to(X.device)
    Z = model.predict(grid).reshape(xx.shape).cpu()

    levels = np.arange(-0.5, model.n_classes + 0.5, 1.0)
    ax.contourf(xx.numpy(), yy.numpy(), Z.numpy(), alpha=0.28, cmap="viridis", levels=levels)
    ax.scatter(X[:, 0].cpu(), X[:, 1].cpu(), c=y.cpu(), cmap="viridis",
               edgecolors="k", linewidths=0.4, s=22)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, k in zip(axes, [1, 5, 25, len(Xtr)]):
    m = TorchKNN(k=k).fit(Xtr, ytr)
    acc = m.score(Xte, yte)
    label = f"k = {k}" + (" (= n)" if k == len(Xtr) else "")
    plot_knn_boundary(ax, m, Xtr, ytr, f"{label}\ntest acc = {acc:.3f}")

plt.tight_layout(); plt.show()
print("k=1 is jagged and overfits. Large k flattens the boundary until every")
print("point is assigned to the majority class.")

### Part E --- K-Means from scratch in PyTorch

Lloyd's algorithm, alternating two steps until the centroids stop moving:

- **E-step:** assign every point to its nearest centroid (`cdist` then `argmin`).
- **M-step:** move every centroid to the mean of its members.

The objective is the within-cluster sum of squares, also called inertia. It can
only decrease, so the algorithm always terminates, though only at a **local**
optimum.

In [ ]:
# ---- user-adjustable parameters ------------------------------------------
TRUE_K = 4          # how many blobs to generate
N_BLOB = 600
CLUSTER_STD = 0.85
# --------------------------------------------------------------------------

Xb_np, yb_np = make_blobs(n_samples=N_BLOB, centers=TRUE_K, n_features=2,
                          cluster_std=CLUSTER_STD, random_state=SEED)

Xb = torch.tensor(Xb_np, dtype=torch.float32).to(device)
yb = torch.tensor(yb_np, dtype=torch.long)          # only for evaluation

print("data:", tuple(Xb.shape), "| true clusters:", TRUE_K)

plt.figure(figsize=(4.5, 3.6))
plt.scatter(Xb[:, 0].cpu(), Xb[:, 1].cpu(), s=14, c="gray")
plt.title("the data, as the algorithm sees it (no labels)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
def kmeans(X, k, n_iter=100, tol=1e-5, seed=0, init="random", verbose=False):
    """Lloyd's algorithm in PyTorch. Returns centroids, assignments, inertia, history."""
    g = torch.Generator(device="cpu").manual_seed(seed)

    if init == "random":
        idx = torch.randperm(X.shape[0], generator=g)[:k]
        centroids = X[idx].clone()
    elif init == "kmeans++":
        centroids = kmeans_plusplus_init(X, k, generator=g)
    else:
        raise ValueError("init must be 'random' or 'kmeans++'")

    history = []
    assign = None

    for it in range(n_iter):
        # ---- E-step: assign each point to the nearest centroid -----------
        D = torch.cdist(X, centroids)              # (n, k)
        assign = D.argmin(dim=1)                   # (n,)

        # ---- M-step: each centroid becomes the mean of its members -------
        new_c = m_step(X, assign, k, fallback=centroids)

        shift = (new_c - centroids).norm().item()
        centroids = new_c

        inertia = (X - centroids[assign]).pow(2).sum().item()
        history.append(inertia)

        if verbose and (it < 3 or it % 10 == 0):
            print(f"  iter {it:3d} | inertia {inertia:12.4f} | shift {shift:.6f}")

        if shift < tol:
            break

    return centroids, assign, history[-1], history


def m_step(X, assign, k, fallback=None):
    """Fully vectorised centroid update, no Python loop over k."""
    n, d = X.shape
    sums = torch.zeros(k, d, device=X.device)
    counts = torch.zeros(k, device=X.device)

    sums.index_add_(0, assign, X)
    counts.index_add_(0, assign, torch.ones(n, device=X.device))

    empty = counts == 0
    new_c = sums / counts.clamp(min=1).unsqueeze(1)
    if fallback is not None and empty.any():
        new_c[empty] = fallback[empty]          # keep old centroid if cluster empty
    return new_c

In [ ]:
def kmeans_plusplus_init(X, k, generator=None):
    """k-means++ seeding: spread the initial centroids out."""
    n = X.shape[0]
    first = torch.randint(0, n, (1,), generator=generator).item()
    centroids = [X[first]]

    for _ in range(1, k):
        C = torch.stack(centroids)
        d2 = torch.cdist(X, C).min(dim=1).values.pow(2)    # distance to nearest centroid
        probs = d2 / d2.sum().clamp(min=1e-12)
        nxt = torch.multinomial(probs.cpu(), 1, generator=generator).item()
        centroids.append(X[nxt])

    return torch.stack(centroids)


centroids, assign, inertia, hist = kmeans(Xb, k=TRUE_K, seed=SEED, verbose=True)

print(f"\nconverged in {len(hist)} iterations, final inertia = {inertia:.4f}")
print("inertia never increases:", all(hist[i] >= hist[i + 1] - 1e-6 for i in range(len(hist) - 1)))
print("cluster sizes:", torch.bincount(assign, minlength=TRUE_K).tolist())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].scatter(Xb[:, 0].cpu(), Xb[:, 1].cpu(), c=assign.cpu(), cmap="viridis", s=16)
axes[0].scatter(centroids[:, 0].cpu(), centroids[:, 1].cpu(),
                marker="X", s=260, c="red", edgecolors="black", linewidths=1.2,
                label="centroids")
axes[0].set_title(f"K-Means result (k={TRUE_K})"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist, "o-")
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("inertia")
axes[1].set_title("the objective only decreases"); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Clustering has no fixed label order, so compare with a permutation-invariant score
print("adjusted Rand index vs the true blobs:",
      round(adjusted_rand_score(yb.numpy(), assign.cpu().numpy()), 4))
print("(1.0 means the partition is identical up to relabelling)")

In [ ]:
# The elbow method: inertia against k.
k_range = list(range(1, 11))
inertias = [kmeans(Xb, k=k, seed=SEED, init="kmeans++")[2] for k in k_range]

plt.figure(figsize=(6, 3.6))
plt.plot(k_range, inertias, "o-")
plt.axvline(TRUE_K, color="red", linestyle="--", alpha=0.7, label=f"true k = {TRUE_K}")
plt.xlabel("k"); plt.ylabel("inertia")
plt.title("elbow method"); plt.legend(); plt.grid(alpha=0.3)
plt.xticks(k_range)
plt.tight_layout(); plt.show()

print("inertia by k:", [round(v, 1) for v in inertias])
print("\nInertia always decreases and hits 0 at k = n, so it cannot simply be")
print("minimised. Look for the bend, where extra clusters stop buying much.")

In [ ]:
# Initialisation matters: run random init from many seeds and look at the spread.
random_runs = [kmeans(Xb, k=TRUE_K, seed=s, init="random")[2] for s in range(15)]
pp_runs     = [kmeans(Xb, k=TRUE_K, seed=s, init="kmeans++")[2] for s in range(15)]

print(f"random   init: best {min(random_runs):.2f} | worst {max(random_runs):.2f} "
      f"| mean {np.mean(random_runs):.2f}")
print(f"kmeans++ init: best {min(pp_runs):.2f} | worst {max(pp_runs):.2f} "
      f"| mean {np.mean(pp_runs):.2f}")

plt.figure(figsize=(6, 3.2))
plt.plot(random_runs, "o-", label="random")
plt.plot(pp_runs, "s-", label="kmeans++")
plt.xlabel("seed"); plt.ylabel("final inertia")
plt.title("final inertia across 15 seeds")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nOn well-separated blobs both inits usually reach the same optimum, so the")
print("spread here may be tiny. Raise CLUSTER_STD, or lower the separation, and")
print("random init starts landing in bad local optima while kmeans++ holds up.")
print("This is why real implementations run several restarts and keep the best.")

In [ ]:
# Where K-Means fails: it assumes roughly spherical clusters of similar size.
from sklearn.datasets import make_moons

Xm_np, ym_np = make_moons(n_samples=400, noise=0.06, random_state=SEED)
Xm = torch.tensor(Xm_np, dtype=torch.float32).to(device)

cm_, am_, im_, _ = kmeans(Xm, k=2, seed=SEED, init="kmeans++")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].scatter(Xm[:, 0].cpu(), Xm[:, 1].cpu(), c=ym_np, cmap="coolwarm", s=16)
axes[0].set_title("true structure (two moons)")
axes[1].scatter(Xm[:, 0].cpu(), Xm[:, 1].cpu(), c=am_.cpu(), cmap="coolwarm", s=16)
axes[1].scatter(cm_[:, 0].cpu(), cm_[:, 1].cpu(), marker="X", s=220,
                c="black", edgecolors="white")
axes[1].set_title("what K-Means finds")
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("K-Means cuts the moons in half because it can only draw straight")
print("boundaries between centroids. For shapes like this, use DBSCAN or")
print("spectral clustering instead.")

In [ ]:
# Everything runs on the GPU without changing a line of the algorithm.
if device.type == "cuda":
    big = torch.randn(200000, 16, device=device)

    torch.cuda.synchronize(); t0 = time.time()
    _ = kmeans(big, k=8, n_iter=20, seed=SEED)
    torch.cuda.synchronize()
    print(f"GPU: 200k points, 16 dims, k=8, 20 iters -> {time.time() - t0:.3f} s")

    big_cpu = big.cpu()
    t0 = time.time()
    _ = kmeans(big_cpu, k=8, n_iter=20, seed=SEED)
    print(f"CPU: same workload                        -> {time.time() - t0:.3f} s")
    print("\nThis is the payoff for writing the algorithm in tensor operations.")
else:
    print("Enable a GPU runtime to see the speed comparison.")

## 3. Challenge

**Challenge 1 --- Distance-weighted KNN.** Instead of every neighbour having one
vote, weight each vote by `1 / (distance + eps)`. Closer neighbours count more.
Does it help on the dataset from Part D?

**Challenge 2 --- Silhouette score.** Implement the silhouette score in PyTorch
and use it to select `k`, as an alternative to the elbow. For a point `i` in
cluster `A`: `a(i)` is its mean distance to other points in `A`, `b(i)` is the
smallest mean distance to any other cluster, and the score is
`(b - a) / max(a, b)`. Higher is better.

**Challenge 3 --- Apply both to Iris.** Run your KNN on the Iris data from Part B
and report the accuracy. Then cluster Iris with `k=3` and compare the partition
to the true species using the adjusted Rand index.

In [ ]:
# TODO Challenge 1
class WeightedKNN(TorchKNN):
    """Override predict() so each neighbour votes with weight 1/(distance+eps)."""

    def predict(self, Q, chunk=4096):
        # your code here: use D.topk(...) to get BOTH indices and values,
        # build the weights, then scatter_add_ the weights instead of ones
        raise NotImplementedError


# TODO Challenge 2
def silhouette_score_torch(X, assign, k):
    # your code here
    raise NotImplementedError

### Solutions

In [ ]:
class WeightedKNN(TorchKNN):
    """KNN where each neighbour votes with weight 1 / (distance + eps)."""

    def __init__(self, k=5, p=2, eps=1e-8):
        super().__init__(k=k, p=p)
        self.eps = eps

    def predict(self, Q, chunk=4096):
        out = []
        for start in range(0, Q.shape[0], chunk):
            q = Q[start:start + chunk]
            D = torch.cdist(q, self.X, p=self.p)
            topk = D.topk(self.k, dim=1, largest=False)
            idx, dist = topk.indices, topk.values           # (m, k) each
            labels = self.y[idx]

            weights = 1.0 / (dist + self.eps)
            votes = torch.zeros(q.shape[0], self.n_classes, device=q.device)
            votes.scatter_add_(1, labels, weights)
            out.append(votes.argmax(dim=1))
        return torch.cat(out)


print("uniform vs distance-weighted voting:")
for k in [1, 5, 11, 21, 45]:
    u = TorchKNN(k=k).fit(Xtr, ytr).score(Xte, yte)
    w = WeightedKNN(k=k).fit(Xtr, ytr).score(Xte, yte)
    print(f"  k={k:3d} | uniform {u:.4f} | weighted {w:.4f}")

print("\nWeighting helps most at large k, where distant, irrelevant neighbours")
print("would otherwise drown out the nearby ones.")

In [ ]:
def silhouette_score_torch(X, assign, k):
    """Mean silhouette over all points. Returns a float in [-1, 1]."""
    n = X.shape[0]
    D = torch.cdist(X, X)                                # (n, n)

    # one-hot membership, shape (n, k)
    onehot = torch.zeros(n, k, device=X.device)
    onehot[torch.arange(n), assign] = 1.0

    sums = D @ onehot                                    # (n, k) sum of distances
    counts = onehot.sum(dim=0)                           # (k,)

    own = assign                                         # cluster of each point
    own_counts = counts[own]

    # a(i): mean distance to OTHER points in the same cluster
    a = sums[torch.arange(n), own] / (own_counts - 1).clamp(min=1)

    # b(i): smallest mean distance to any other cluster
    mean_to_cluster = sums / counts.clamp(min=1).unsqueeze(0)
    mean_to_cluster[torch.arange(n), own] = float("inf")
    b = mean_to_cluster.min(dim=1).values

    s = (b - a) / torch.maximum(a, b).clamp(min=1e-12)
    s[own_counts == 1] = 0.0                             # singleton clusters score 0
    return s.mean().item()


sil = []
for k in range(2, 11):
    _, asg, _, _ = kmeans(Xb, k=k, seed=SEED, init="kmeans++")
    sil.append(silhouette_score_torch(Xb, asg, k))

plt.figure(figsize=(6, 3.4))
plt.plot(range(2, 11), sil, "o-", color="seagreen")
plt.axvline(TRUE_K, color="red", linestyle="--", alpha=0.7, label=f"true k = {TRUE_K}")
plt.xlabel("k"); plt.ylabel("mean silhouette")
plt.title("silhouette method"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

best_k = int(np.argmax(sil)) + 2
print("silhouette by k:", [round(v, 4) for v in sil])
print(f"best k by silhouette = {best_k} (true k = {TRUE_K})")
print("\nUnlike inertia, the silhouette has a genuine maximum, so it can be")
print("optimised directly rather than eyeballed.")

In [ ]:
# Challenge 3: both algorithms on the Iris data from Part B.

X_all = torch.cat([iris_train.X, iris_val.X]).to(device)
y_all = torch.cat([iris_train.y, iris_val.y])

# --- KNN ------------------------------------------------------------------
iris_knn = TorchKNN(k=5).fit(iris_train.X.to(device), iris_train.y.to(device))
iris_acc = iris_knn.score(iris_val.X.to(device), iris_val.y.to(device))
print(f"KNN (k=5) on Iris: validation accuracy = {iris_acc:.4f}")

for k in [1, 3, 5, 7, 11, 15]:
    m = TorchKNN(k=k).fit(iris_train.X.to(device), iris_train.y.to(device))
    print(f"  k={k:2d} -> {m.score(iris_val.X.to(device), iris_val.y.to(device)):.4f}")

# --- K-Means --------------------------------------------------------------
c_iris, a_iris, in_iris, _ = kmeans(X_all, k=3, seed=SEED, init="kmeans++")
ari = adjusted_rand_score(y_all.numpy(), a_iris.cpu().numpy())

print(f"\nK-Means (k=3) on Iris: inertia = {in_iris:.3f}, ARI vs true species = {ari:.4f}")
print("cluster sizes:", torch.bincount(a_iris, minlength=3).tolist())
print("true  sizes  :", torch.bincount(y_all, minlength=3).tolist())
print("\nsetosa separates cleanly; versicolor and virginica overlap, which caps the ARI.")

In [ ]:
# Visualise the Iris comparison on the two most informative features.
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

axes[0].scatter(X_all[:, 2].cpu(), X_all[:, 3].cpu(), c=y_all, cmap="viridis", s=22)
axes[0].set_title("true species (supervised labels)")

axes[1].scatter(X_all[:, 2].cpu(), X_all[:, 3].cpu(), c=a_iris.cpu(), cmap="viridis", s=22)
axes[1].scatter(c_iris[:, 2].cpu(), c_iris[:, 3].cpu(), marker="X", s=240,
                c="red", edgecolors="black")
axes[1].set_title("K-Means clusters (no labels used)")

for ax in axes:
    ax.set_xlabel("petal_length (scaled)"); ax.set_ylabel("petal_width (scaled)")
    ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Recap

| Concept | Key point |
|---|---|
| `Dataset` | Implement `__len__` and `__getitem__`, nothing else |
| Lazy loading | Store paths in `__init__`, load in `__getitem__` |
| `DataLoader` | `batch_size`, `shuffle` (train only), `num_workers` |
| Transform order | `ToTensor()` before `Normalize()` |
| Normalisation | Compute statistics on the training split only |
| Image layout | `(N, C, H, W)` in torch, `.permute(1, 2, 0)` for matplotlib |
| EDA | shape, dtype, range, NaN, class balance, then plot |
| `torch.cdist` | One call gives the full `(m, n)` distance matrix |
| KNN | `cdist` then `topk(largest=False)` then vote |
| Choosing k (KNN) | Small k overfits, large k underfits; cross-validate |
| K-Means | E-step `argmin`, M-step `index_add_`, repeat |
| Choosing k (K-Means) | Elbow on inertia, or maximise the silhouette |
| Initialisation | k-means++ and several restarts |

### End of the course

You have written, from scratch and in PyTorch: a neural network, a data pipeline,
a classifier and a clustering algorithm. The natural next steps are convolutional
networks, transfer learning with `torchvision.models`, and a project of your own.